In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.0/112.6 GB disk)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!find /content/drive/MyDrive -maxdepth 3 -type d -iname "*Rock2026812*"

/content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11


In [ ]:
import os, yaml
DATASET_DIR = "/content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11"
DATA_YAML   = os.path.join(DATASET_DIR, "data.yaml")

print(os.listdir(DATASET_DIR))

d = yaml.safe_load(open(DATA_YAML))
d["path"], d["train"], d["val"], d["test"] = DATASET_DIR, "train/images", "valid/images", "test/images"
yaml.safe_dump(d, open(DATA_YAML, "w"), sort_keys=False)
print(d)

for s in ["train", "valid", "test"]:
    p = os.path.join(DATASET_DIR, s, "images")
    print(s, len(os.listdir(p)) if os.path.exists(p) else "lose")

['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'train', 'valid', 'test']
{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['rock'], 'roboflow': {'workspace': 'chuyaos-workspace-agyyw', 'project': 'rock2026812', 'version': 7, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/chuyaos-workspace-agyyw/rock2026812/dataset/7'}, 'path': '/content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11'}
train 359
valid 74
test 76


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
results = model.train(
    data=DATA_YAML,
    epochs=100, imgsz=640, batch=16,
    optimizer="auto", seed=42, deterministic=True,
    patience=20, amp=True,
    degrees=20, fliplr=0.5, flipud=0.5, scale=0.5, translate=0.1,
    shear=3.0, perspective=0.0005,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.4,
    mosaic=1.0, close_mosaic=10, mixup=0.15, copy_paste=0.1, erasing=0.4,
    project="runs", name="yolo11s_rock_v7",
)

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11/data.yaml, degrees=20, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_

In [ ]:
metrics = model.val(data=DATA_YAML, split="test")
print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}  P={metrics.box.mp:.3f}  R={metrics.box.mr:.3f}")

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.2 ms, read: 4.5±1.8 MB/s, size: 2379.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11/test/labels... 76 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 76/76 1.4s/it 1:47
val: New cache created: /content/drive/MyDrive/Rock2026812.v7-rock_detection_509_v3.yolov11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 7.0s/it 34.8s
                   all         76         97      0.876      0.802      0.912      0.522
Speed: 1.9ms preprocess, 12.3ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to

In [ ]:
from pathlib import Path

best = Path(results.save_dir) / "weights" / "best.pt"
!cp "{best}" "/content/drive/MyDrive/best_yolo11s_v7_comprehensive.pt"
print("saved:", best)

saved: /content/runs/detect/runs/yolo11s_rock_v7/weights/best.pt
